# Vision Hallucination Metrics

Evaluate a Vision Language Model (VLM) for hallucination in detection systems.

An LLM judge compares the VLM's free-text description against the ground truth and classifies each prediction as `true_positive`, `false_positive`, `true_negative`, or `false_negative`.

## Metrics

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **FalsePositiveRate** | FP / (FP + TN) | How often the model invents events that never occurred. A high FPR floods operators with false alarms and destroys trust. |
| **Precision** | TP / (TP + FP) | Of all the alerts raised, how many were real. Low precision means most alerts are false — operators start ignoring them. |
| **ConfidenceScoreAnalysis** | ECE + distribution stats | Whether the model's confidence scores are reliable for threshold tuning. ECE measures the gap between "how confident the model says it is" and "how often it's actually right". A high ECE means confidence scores can't be used directly — recalibration is needed before production. |

In [ ]:
!pip install alquimia-fair-forge langchain-groq

In [ ]:
import json
import logging
import getpass
from pathlib import Path

from langchain_groq import ChatGroq

from fair_forge import Retriever
from fair_forge.metrics.vision import ConfidenceScoreAnalysis, FalsePositiveRate, Precision
from fair_forge.schemas import Dataset

logging.getLogger('httpx').disabled = True
logging.getLogger('httpcore').disabled = True

## Setup

This example uses [Groq](https://console.groq.com) — a fast, free LLM API.

In [ ]:
GROQ_API_KEY = getpass.getpass('Enter your Groq API key: ')

judge_model = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=GROQ_API_KEY,
    temperature=0.0,
)


class VLMRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path('dataset.json'), encoding='utf-8') as f:
            return [Dataset.model_validate(item) for item in json.load(f)]

## False Positive Rate

How often does the model invent events that never occurred?

**FPR = FP / (FP + TN)**

In [ ]:
fpr_results = FalsePositiveRate.run(VLMRetriever, model=judge_model)

for r in fpr_results:
    fpr = r.false_positive_rate
    print(f'{r.session_id}')
    print(f'  FPR            : {fpr:.1%}' if fpr is not None else '  FPR            : N/A (no actual negatives)')
    print(f'  False alarms   : {r.n_false_positives} / {r.n_negatives} actual negative frames')
    print(f'  Total frames   : {r.n_predictions}')
    print()
    for i in r.interactions:
        if i.classification == 'false_positive':
            print(f'  [FP] {i.qa_id}: {i.reasoning}')
    print()

## Precision

When the model raised an alert, how often was it correct?

**Precision = TP / (TP + FP)**

In [ ]:
precision_results = Precision.run(VLMRetriever, model=judge_model)

for r in precision_results:
    prec = r.precision
    print(f'{r.session_id}')
    print(f'  Precision      : {prec:.1%}' if prec is not None else '  Precision      : N/A (no positive predictions)')
    print(f'  True alerts    : {r.n_true_positives} / {r.n_positive_predictions} total alerts raised')
    print(f'  Total frames   : {r.n_predictions}')
    print()

## Confidence Score Analysis

Are the VLM's confidence scores calibrated? A score of 0.8 should mean the model is right ~80% of the time.

Reads `Batch.agentic["confidence"]` and computes distribution stats + Expected Calibration Error (ECE).

In [ ]:
confidence_results = ConfidenceScoreAnalysis.run(VLMRetriever, model=judge_model)

for r in confidence_results:
    print(f'{r.session_id}')
    print(f'  Frames with confidence : {r.n_with_confidence} / {r.n_predictions}')
    if r.confidence_mean is not None:
        print(f'  Confidence             : mean={r.confidence_mean:.2f}  std={r.confidence_std:.2f}  min={r.confidence_min:.2f}  max={r.confidence_max:.2f}')
    ece = r.expected_calibration_error
    print(f'  ECE                    : {ece:.3f}' if ece is not None else '  ECE                    : N/A')
    print()
    non_empty = [b for b in r.buckets if b.count > 0]
    if non_empty:
        print(f'  {"Range":<12} {"n":<6} {"Mean conf":<12} {"Accuracy":<10} {"Gap"}')
        print(f'  {"-" * 50}')
        for b in non_empty:
            acc = f'{b.accuracy:.0%}' if b.accuracy is not None else 'N/A'
            conf = f'{b.mean_confidence:.0%}' if b.mean_confidence is not None else 'N/A'
            gap = f'{abs(b.mean_confidence - b.accuracy):.2f}' if b.mean_confidence is not None and b.accuracy is not None else ''
            print(f'  [{b.range_min:.1f}–{b.range_max:.1f}]    {b.count:<6} {conf:<12} {acc:<10} {gap}')
    print()